# Sector HMM Detection

Notebook này sử dụng dữ liệu `sector_hmm_data.csv` đã qua chuẩn hóa NQT để nhận diện trạng thái (Regime) của từng nhóm ngành (Sector).

In [1]:
import numpy as np
import pandas as pd
import warnings
from hmmlearn.hmm import GMMHMM
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(42)

OUTPUT_DIR = Path('../output/sector_hmm')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Thư mục đầu ra: {OUTPUT_DIR.resolve()}")

Thư mục đầu ra: C:\Users\ADMIN\Desktop\Kaggle\output\sector_hmm


In [2]:
print("Tải dữ liệu Sector HMM...")
df_sector = pd.read_csv('../data/processed/sector_hmm_data.csv')
df_sector['time'] = pd.to_datetime(df_sector['time'])

Z_features = ['sector_log_ret_Z', 'sector_vol20_Z', 'sector_vol5_Z', 'sector_volume_ratio_Z']
print(f"Kích thước dữ liệu: {df_sector.shape}")

Tải dữ liệu Sector HMM...
Kích thước dữ liệu: (52822, 12)


In [3]:
def auto_label(rs, K):
    ret = rs['mean_ret'].values
    vol = rs['mean_vol'].values
    if K == 2:
        return {int(np.argmin(ret)): 'Bear', int(np.argmax(ret)): 'Bull'}
    elif K == 3:
        order = np.argsort(ret)
        bear_idx = order[0]
        rem = [order[1], order[2]]
        if vol[rem[0]] < vol[rem[1]]:
            bull_idx, side_idx = rem[0], rem[1]
        else:
            bull_idx, side_idx = rem[1], rem[0]
        return {int(bear_idx): 'Bear', int(side_idx): 'Sideways', int(bull_idx): 'Bull'}
    return {i: f'State_{i}' for i in range(K)}

results = []
K_sector = 3 # Cố định K=3 (Bull, Bear, Sideways) để làm mẫu

print("Bắt đầu huấn luyện HMM cho từng ngành...")
for industry, group in df_sector.groupby('industry'):
    group = group.sort_values('time').reset_index(drop=True)
    Z_data = group[Z_features].fillna(0).values
    
    if len(Z_data) < 100:
        print(f"Bỏ qua {industry} do quá ít dữ liệu.")
        continue
        
    best_model, best_ll = None, -np.inf
    for seed in range(3):
        try:
            model = GMMHMM(n_components=K_sector, n_mix=2, covariance_type='full', n_iter=100, random_state=seed*7)
            model.fit(Z_data)
            ll = model.score(Z_data)
            if ll > best_ll:
                best_ll = ll
                best_model = model
        except:
            continue
            
    if best_model is None:
        print(f"Lỗi: Không thể hội tụ cho ngành {industry}")
        continue
        
    states = best_model.predict(Z_data)
    probs = best_model.predict_proba(Z_data)
    
    group['sector_regime'] = states
    for k in range(K_sector):
        group[f'prob_sector_{k}'] = probs[:, k]
        
    # Gán nhãn tự động theo rủi ro và lợi nhuận cho từng nhóm ngành
    stats = []
    for k in range(K_sector):
        mask = group['sector_regime'] == k
        r = group.loc[mask, 'sector_log_ret'].mean()
        v = group.loc[mask, 'sector_vol20'].mean()
        stats.append({'state': k, 'mean_ret': r, 'mean_vol': v})
        
    labels = auto_label(pd.DataFrame(stats), K_sector)
    group['sector_regime_label'] = group['sector_regime'].map(labels)
    
    results.append(group)
    print(f"[+] Hoàn thành: {industry} (Log-Likelihood: {best_ll:.2f})")

df_final_sectors = pd.concat(results, ignore_index=True)
df_final_sectors.to_csv(OUTPUT_DIR / 'master_sector_regimes.csv', index=False)
print(f"\nĐã lưu toàn bộ trạng thái của {len(results)} ngành vào: {OUTPUT_DIR / 'master_sector_regimes.csv'}")

Bắt đầu huấn luyện HMM cho từng ngành...
[+] Hoàn thành: Bán lẻ (Log-Likelihood: -11141.11)
[+] Hoàn thành: Bảo hiểm (Log-Likelihood: -11260.88)
[+] Hoàn thành: Bất động sản (Log-Likelihood: -11182.19)
[+] Hoàn thành: Bất động sản KCN (Log-Likelihood: -11239.76)
[+] Hoàn thành: Cao su (Log-Likelihood: -11303.27)
[+] Hoàn thành: Chứng khoán (Log-Likelihood: -11128.22)
[+] Hoàn thành: Công nghệ (Log-Likelihood: -11052.45)
[+] Hoàn thành: Cảng biển (Log-Likelihood: -11230.56)
[+] Hoàn thành: Dầu khí (Log-Likelihood: -11075.24)
[+] Hoàn thành: Hạ tầng (Log-Likelihood: -11127.25)


Model is not converging.  Current: -9932.827597617887 is not greater than -9774.208483525776. Delta is -158.61911409211098


[+] Hoàn thành: Ngân hàng (Log-Likelihood: -11115.20)
[+] Hoàn thành: Nhựa (Log-Likelihood: -11331.67)
[+] Hoàn thành: Nông nghiệp (Log-Likelihood: -11283.55)
[+] Hoàn thành: Năng lượng (Log-Likelihood: -11107.59)
[+] Hoàn thành: Phân bón (Log-Likelihood: -11097.63)
[+] Hoàn thành: Thép (Log-Likelihood: -11185.44)
[+] Hoàn thành: Thủy sản (Log-Likelihood: -11104.82)
[+] Hoàn thành: Thực phẩm (Log-Likelihood: -11202.40)
[+] Hoàn thành: Vận tải dầu khí (Log-Likelihood: -11259.44)
[+] Hoàn thành: Vật liệu xây dựng (Log-Likelihood: -10993.00)
[+] Hoàn thành: Xây dựng (Log-Likelihood: -11149.69)
[+] Hoàn thành: Điện (Log-Likelihood: -11097.84)

Đã lưu toàn bộ trạng thái của 22 ngành vào: ..\output\sector_hmm\master_sector_regimes.csv
